<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 9A · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">保留输入、分流拒收、自动验收</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">使用统一订单样本，观察 SQL、结果与验收证据。请按顺序运行单元。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

完成后，你将把十三行原始输入分为十行合格订单与三行拒收记录，核对拒收原因，并用故意注入的错误检验质量规则。请按顺序运行。

[讲义](course9a_data_quality_and_schema_validation.md) · [课程入口](../README.md)


## 实验范围

先完成 D05，确认 wwi_customers 存在；本节读取该历史客户表，不修改它。

重建 orders_raw、customers、orders_classified 视图、orders_clean 和 orders_rejected。13 行输入包括三种不同的异常，先全部暂存，再做业务准入。D06 读取本 Lab 的合格输出。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = WarehouseLab()




## 1. 保留模拟新订单的原始字段

金额、订单号和客户号先作为字符串暂存。13 行输入包含 10 笔正常新订单，以及非法金额、缺失订单号、无效客户各一行。
客户维度读取 D05 已导入的 wwi_customers（663 行），而不是给每条输入临时造一个客户。与 D05 的整批拒绝不同，本节保留原始输入再分流。


In [ ]:
lab.execute("DROP VIEW IF EXISTS orders_classified")
lab.execute("DROP TABLE IF EXISTS orders_raw")
fields = ", ".join(col + " VARCHAR(100) NULL" for col in ORDER_COLUMNS)
lab.execute(f'CREATE TABLE orders_raw (input_id BIGINT NOT NULL, {fields}) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
raw = fixture("raw_orders.json")
lab.insert("orders_raw", ["input_id", *ORDER_COLUMNS],
           [(r["input_id"], *[None if r[col] is None else str(r[col]) for col in ORDER_COLUMNS]) for r in raw])
expect(lab.query("SELECT COUNT(*) FROM orders_raw"), [(13,)])

lab.execute("DROP TABLE IF EXISTS customers")
lab.execute('CREATE TABLE customers (customer_id BIGINT NOT NULL, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.execute("INSERT INTO customers SELECT CustomerID, CustomerName FROM wwi_customers")
expect(lab.query("SELECT COUNT(*) FROM customers"), [(663,)])


## 2. 按固定规则分流

规则依次判断订单号、金额、客户引用和来源；本批数据中三条异常分别落入前三类。
缺失或格式不合法的客户号也应拒收。范围是本节公布的数据契约，不声称覆盖所有生产数据质量规则。


In [ ]:
lab.execute("""
CREATE VIEW orders_classified AS
SELECT *, CASE
    WHEN TRY_CAST(order_id AS BIGINT) IS NULL THEN 'INVALID_ORDER_ID'
    WHEN TRY_CAST(order_amount AS DECIMAL(12,2)) IS NULL
      OR TRY_CAST(order_amount AS DECIMAL(12,2)) < 0 THEN 'INVALID_AMOUNT'
    WHEN TRY_CAST(customer_id AS BIGINT) IS NULL THEN 'INVALID_CUSTOMER'
    WHEN TRY_CAST(customer_id AS BIGINT) NOT IN (SELECT customer_id FROM customers) THEN 'INVALID_CUSTOMER'
    WHEN data_source IS NULL OR data_source <> 'COURSE_SIMULATION' THEN 'INVALID_SOURCE'
    ELSE NULL END AS reject_reason
FROM orders_raw
""")
lab.execute("DROP TABLE IF EXISTS orders_clean")
ddl = order_ddl("orders_clean")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS orders_rejected")
lab.execute('CREATE TABLE orders_rejected (input_id BIGINT, reason VARCHAR(32)) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")
lab.execute("INSERT INTO orders_rejected SELECT input_id, reject_reason FROM orders_classified WHERE reject_reason IS NOT NULL")


## 3. 检查业务质量并注入错误

数量只是第一层校验，还要比对完整记录。事件时间检查用固定截止时刻，不冒充真实端到端接入延迟。


In [ ]:
expected_rows = order_rows(fixture("orders.json"))
def quality_report():
    expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")])
    expect(lab.query("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders_clean"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE event_time > '2026-01-02 12:00:00' OR event_time IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE status <> 'CREATED' OR paid_amount <> 0 OR refund_amount <> 0"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE data_source <> 'COURSE_SIMULATION'"), [(0,)])
    # DATE_FORMAT makes the transport representation explicit for fixture comparison.
    projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
    expect(lab.query(f"SELECT {projection} FROM orders_clean ORDER BY order_id"), expected_rows)

expect(lab.query("SELECT input_id, reason FROM orders_rejected ORDER BY input_id"),
       [(11,"INVALID_AMOUNT"),(12,"INVALID_ORDER_ID"),(13,"INVALID_CUSTOMER")])
expect(lab.query("SELECT COUNT(*) FROM orders_classified WHERE reject_reason IS NULL"), [(10,)])
expect(lab.query("SELECT COUNT(*) FROM orders_raw r LEFT JOIN orders_rejected x ON r.input_id=x.input_id WHERE x.input_id IS NULL"), [(10,)])
quality_report()

# Inject one duplicate: the validator must detect it, not just print a warning.
lab.insert("orders_clean", ORDER_COLUMNS, [expected_rows[0]])
try:
    quality_report()
except AssertionError as error:
    print("Expected quality failure:", error)
else:
    raise AssertionError("Injected duplicate was not detected")
lab.execute("TRUNCATE TABLE orders_clean")
lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")
quality_report()
lab.close()


## 完成与自己动手

13 行输入、10 行合格、3 行拒收。合格订单逐行匹配，拒收原因可回查原始字段。
解释为什么客户号能转成整数，仍可能不应进入数仓。D06 只能从 orders_clean 开始，不绕过质量准入。
十笔正常新订单引用 WWI 客户；全部状态、价格、地区和事件时间都是课程模拟，不回填到历史 WWI 表。
